# 01B · Tradução dos Códigos das Tabelas


## 0. Configuração do Ambiente

In [1]:
import sys
import os
import pandas as pd
from pathlib import Path

In [2]:
# Garante que a raiz do projeto está no sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [3]:
# Importa utilitários do projeto
from src.utils.download_data_from_datasus import download_data, download_dicionarios
from src.utils.converter_dbc_para_csv import converter_dbc_para_csv_lote

print("Utilitários importados")

Utilitários importados


## 4. Tradução dos Códigos das Tabelas

O DataSUS armazena os dados como **códigos numéricos** (ex: `1`, `3`, `M`). Esta seção
traduz esses códigos para **descrições legíveis** usando os dicionários `.cnv` e as
instruções `.def` baixados do FTP do DataSUS.

**Como funciona:**
1. O arquivo `.def` indica quais colunas têm código e qual `.cnv` usar para cada uma
2. O arquivo `.cnv` contém o mapeamento `CÓDIGO → DESCRIÇÃO`
3. Para cada coluna com dicionário é criada uma coluna `COLUNA_DESC` com o texto traduzido

**Resultado:** arquivos salvos em `data/interim/` com colunas `_DESC` adicionadas.

### 4.0 Download dos dicionários (executar apenas uma vez)

Os dicionários `.def` e `.cnv` são baixados do FTP do DataSUS e salvos em `data/external/`.
Se os arquivos já existirem localmente este passo pode ser pulado.

In [4]:
# Pasta destino ancorada em ROOT para não depender do diretório de trabalho do notebook
PASTA_EXTERNAL_SIH = Path(ROOT, "data", "external", "SIH")
PASTA_EXTERNAL_CNES = Path(ROOT, "data", "external", "CNES")
PASTA_EXTERNAL_SIH.mkdir(parents=True, exist_ok=True)
PASTA_EXTERNAL_CNES.mkdir(parents=True, exist_ok=True)

PASTA_INPUT_CNES = Path(ROOT, "data", "input", "CNES")
PASTA_INPUT_SIH= Path(ROOT, "data", "input", "SIH")

In [ ]:
# Dicionários do CNES
#download_dicionarios("CNES", str(PASTA_EXTERNAL_CNES))

# Dicionários do SIH
#download_dicionarios("SIH", str(PASTA_EXTERNAL_SIH))


### 4.1 Configuração dos caminhos e importação dos utilitários

In [5]:
from src.utils.information_translation import mapear_colunas_def, traduzir_csv_datasus

# Caminhos dos dicionários — separados por sistema (CNES e SIH)
PASTA_DEF_CNES  = PASTA_EXTERNAL_CNES                    # arquivos .def do CNES
PASTA_CNV_CNES  = PASTA_EXTERNAL_CNES / "CNV"            # arquivos .cnv do CNES
PASTA_DEF_SIH   = PASTA_EXTERNAL_SIH                     # arquivos .def do SIH
PASTA_CNV_SIH   = PASTA_EXTERNAL_SIH / "CNV"             # arquivos .cnv do SIH

# Pasta de saída dos CSVs traduzidos
PASTA_INTERIM = Path(ROOT, "data", "interim")
PASTA_INTERIM.mkdir(parents=True, exist_ok=True)

print(f"DEF CNES : {PASTA_DEF_CNES}")
print(f"CNV CNES : {PASTA_CNV_CNES}")
print(f"DEF SIH  : {PASTA_DEF_SIH}")
print(f"CNV SIH  : {PASTA_CNV_SIH}")
print(f"Saída    : {PASTA_INTERIM}")

DEF CNES : /home/carolina/Documents/TCC Documentos/TCC/data/external/CNES
CNV CNES : /home/carolina/Documents/TCC Documentos/TCC/data/external/CNES/CNV
DEF SIH  : /home/carolina/Documents/TCC Documentos/TCC/data/external/SIH
CNV SIH  : /home/carolina/Documents/TCC Documentos/TCC/data/external/SIH/CNV
Saída    : /home/carolina/Documents/TCC Documentos/TCC/data/interim


### 4.2 Tradução de Teste (1 arquivo por tipo)
Rode esta célula para testar rapidamente a tradução em apenas 1 arquivo de cada tabela. Isso permite verificar os resultados antes de processar toda a base.


In [6]:
# Mapeamento CNES
MAPA_DEF_CNES = {
    'hb': 'Habilitacao.def',
    'lt': 'Leitos_Especialidade.def',
    'eq': 'Equipamento.def',
    'sr': 'Servico_Especializado_200803_.def',
    'st': 'Estabelecimento.def',
}

print('--- TESTE DE TRADUÇÃO ---')
for prefixo, nome_def in MAPA_DEF_CNES.items():
    csvs = sorted(PASTA_INPUT_CNES.glob(f'{prefixo}*.csv'))
    if not csvs: continue
    
    arquivo_def = PASTA_DEF_CNES / nome_def
    mapa = mapear_colunas_def(str(arquivo_def))
    if not mapa: continue
    
    csv_teste = csvs[0] # Pega apenas o primeiro
    caminho_saida = PASTA_INTERIM / f'cnes_{prefixo}_{csv_teste.stem}_traduzido_teste.csv'
    import json as json_lib
    path_dict_cnes = Path(ROOT, 'data', 'external', f'dicionario_CNES_{prefixo.upper()}.json')
    rename_dict_cnes = None
    if path_dict_cnes.exists():
        with open(path_dict_cnes, 'r', encoding='utf-8') as f:
            schema_cnes = json_lib.load(f)
        rename_dict_cnes = {col['old_name']: col['new_name'] for col in schema_cnes}
    traduzir_csv_datasus(str(csv_teste), mapa, str(PASTA_CNV_CNES), str(caminho_saida), dicionario_renomeacao=rename_dict_cnes)

# Mapeamento SIH
defs_sih = list(PASTA_DEF_SIH.glob('RD*.def')) + list(PASTA_DEF_SIH.glob('RD*.DEF'))
if defs_sih:
    mapa_sih = mapear_colunas_def(str(defs_sih[0]))
    csvs_sih = sorted(PASTA_INPUT_SIH.glob('*.csv'))
    if csvs_sih:
        csv_teste = csvs_sih[0]
        caminho_saida = PASTA_INTERIM / f'sih_{csv_teste.stem}_traduzido_teste.csv'
        import json as json_lib
        with open(Path(ROOT, 'data', 'external', 'dicionario_SIH.json'), 'r', encoding='utf-8') as f:
            schema = json_lib.load(f)
        rename_dict_sih = {col['old_name']: col['new_name'] for col in schema}
        traduzir_csv_datasus(str(csv_teste), mapa_sih, str(PASTA_CNV_SIH), str(caminho_saida), dicionario_renomeacao=rename_dict_sih)


--- TESTE DE TRADUÇÃO ---

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_hb_hbsp1501_traduzido_teste.csv

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_lt_ltsp1501_traduzido_teste.csv

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_eq_eqsp1501_traduzido_teste.csv

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_sr_srsp1501_traduzido_teste.csv


/home/carolina/Documents/TCC Documentos/TCC/src/utils/information_translation.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna_csv] = pd.to_datetime(valores_limpos, format='%Y%m%d', errors='coerce').dt.strftime('%Y-%m-%d').fillna(df[coluna_csv])
/home/carolina/Documents/TCC Documentos/TCC/src/utils/information_translation.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna_csv] = pd.to_datetime(valores_limpos, format='%Y%m%d', errors='coerce').dt.strftime('%Y-%m-%d').fillna(df[coluna_csv])
/home/carolina/Documents/TCC Documentos/


Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_st_stsp1501_traduzido_teste.csv

Sucesso! O arquivo traduzido foi salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_rdsp1501_traduzido_teste.csv


### 4.3 Tradução Completa em Lote (Multiprocessamento - CPU)
Rode esta célula para processar **todos os arquivos** simultaneamente usando os 16 núcleos do seu processador.


In [ ]:
import concurrent.futures

def processar_arquivo(args):
    caminho_csv, mapa, pasta_cnv, caminho_saida, rename_dict = args
    if not caminho_saida.exists():
        traduzir_csv_datasus(caminho_csv, mapa, pasta_cnv, str(caminho_saida), dicionario_renomeacao=rename_dict)
    return caminho_saida.name

tarefas = []

# Prepara tarefas CNES
for prefixo, nome_def in MAPA_DEF_CNES.items():
    csvs = sorted(PASTA_INPUT_CNES.glob(f'{prefixo}*.csv'))
    arquivo_def = PASTA_DEF_CNES / nome_def
    if not arquivo_def.exists(): continue
    mapa = mapear_colunas_def(str(arquivo_def))
    if not mapa: continue
    
    path_dict_cnes = Path(ROOT, 'data', 'external', f'dicionario_CNES_{prefixo.upper()}.json')
    rename_dict_cnes = None
    if path_dict_cnes.exists():
        import json as json_lib
        with open(path_dict_cnes, 'r', encoding='utf-8') as f:
            schema_cnes = json_lib.load(f)
        rename_dict_cnes = {col['old_name']: col['new_name'] for col in schema_cnes}
    
    for csv_path in csvs:
        caminho_saida = PASTA_INTERIM / f'cnes_{prefixo}_{csv_path.stem}_traduzido.csv'
        tarefas.append((str(csv_path), mapa, str(PASTA_CNV_CNES), caminho_saida, rename_dict_cnes))

# Prepara tarefas SIH
if defs_sih:
    mapa_sih = mapear_colunas_def(str(defs_sih[0]))
    import json as json_lib
    with open(Path(ROOT, 'data', 'external', 'dicionario_SIH.json'), 'r', encoding='utf-8') as f:
        schema = json_lib.load(f)
    rename_dict_sih = {col['old_name']: col['new_name'] for col in schema}
    for csv_path in sorted(PASTA_INPUT_SIH.glob('*.csv')):
        caminho_saida = PASTA_INTERIM / f'sih_{csv_path.stem}_traduzido.csv'
        tarefas.append((str(csv_path), mapa_sih, str(PASTA_CNV_SIH), caminho_saida, rename_dict_sih))

print(f'Iniciando tradução de {len(tarefas)} arquivos utilizando multiprocessamento...')

with concurrent.futures.ProcessPoolExecutor() as executor:
    resultados = list(executor.map(processar_arquivo, tarefas))

print(f'\n✓ Tradução completa finalizada! {len(resultados)} arquivos salvos na pasta interim.')


### 4.4 Verificação do resultado

Mostra as colunas `_DESC` geradas em um arquivo de exemplo para confirmar que a tradução funcionou.

In [ ]:
arquivos_traduzidos = sorted(PASTA_INTERIM.glob("*.csv"))
print(f"{len(arquivos_traduzidos)} arquivo(s) traduzido(s) em {PASTA_INTERIM}\n")

if arquivos_traduzidos:
    exemplo = arquivos_traduzidos[0]
    df_ex = pd.read_csv(exemplo, nrows=3, low_memory=False)
    colunas_desc = [c for c in df_ex.columns if c.endswith("_DESC")]
    print(f"Exemplo: {exemplo.name}")
    print(f"Colunas _DESC geradas ({len(colunas_desc)}): {colunas_desc}\n")
    display(df_ex[colunas_desc].head(3))


## Preview dos Dados Coletados

In [ ]:
# Preview do primeiro arquivo CNES (ST) traduzido
csvs_cnes_st_traduzidos = sorted(PASTA_INTERIM.glob("cnes_st*.csv"))
if csvs_cnes_st_traduzidos:
    df_preview_cnes = pd.read_csv(csvs_cnes_st_traduzidos[0], nrows=5, low_memory=False)
    print(f"CNES/ST Traduzido — {csvs_cnes_st_traduzidos[0].name}: {df_preview_cnes.shape[0]} linhas (amostra) × {df_preview_cnes.shape[1]} colunas")
    display(df_preview_cnes.head())
else:
    print("Nenhum arquivo CNES/ST traduzido encontrado em", PASTA_INTERIM)
